In [1]:
# 代码作用：读取MES、LIMS数据汇总为C糖煮制(班报) v1
# 2026/01/21 
# 数据治理工程师：yushumeng
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, DecimalType,DateType
from pyspark.sql.functions import current_timestamp,split,lit,col
from pyspark.sql.functions import col, lit, sum, avg, count, when, round,concat
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import StringType, DecimalType
from pyspark.sql.utils import AnalysisException
from pyspark.sql.functions import to_date
import warnings
warnings.filterwarnings("ignore")

print("Spark 启动中...")
spark = SparkSession.builder \
    .appName("lims_dwd") \
    .enableHiveSupport() \
    .getOrCreate()

# 读取MES数据
mes_sql = """
WITH ranked_data AS (
    SELECT 
        T1.season,
        T1.factory,
        T1.hand_date,
        T1.team,
        T2.project,
        CASE WHEN T2.value REGEXP '^[0-9.]+$' THEN CAST(T2.value AS DECIMAL(10,2)) ELSE NULL END AS value,
        ROW_NUMBER() OVER (
            PARTITION BY T1.season, T1.factory, T1.hand_date, T1.team, T2.project
            ORDER BY T1.hand_time DESC   -- 改为降序，取最晚记录
        ) AS rn
    FROM dwd.dwd_mes_pro_shift_change_record_f_1h T1 
    INNER JOIN dwd.dwd_mes_pro_shift_change_detail_f_1h T2 
        ON T1.hand_code = T2.record_code
    WHERE T2.project IN ("当班原料用量(m³)：","当班放糖量(m³)：","当班种子用量(m³)：") 
      AND T1.flag_deleted = 0
),
unique_data AS (
    SELECT 
        season,
        factory,
        hand_date,
        team,
        project,
        value
    FROM ranked_data
    WHERE rn = 1
)
SELECT 
    season,
    factory,
    hand_date,
    team,
    MAX(CASE WHEN project = '当班原料用量(m³)：' THEN value END) AS raw_material_vol_m3,
    MAX(CASE WHEN project = '当班放糖量(m³)：' THEN value END) AS sugar_output_vol_m3,
    MAX(CASE WHEN project = '当班种子用量(m³)：' THEN value END) AS seed_vol_m3
FROM unique_data
GROUP BY season, factory, hand_date, team
ORDER BY season, factory, hand_date, team;
"""
mes_df = spark.sql(mes_sql)

# 读取LIMS样品丙膏数据
lims_sql = """
WITH raw_data AS (
    SELECT 
        ord_accept_org_code AS company,
        ord_crushing_season AS crushing_season,
        ord_accept_date AS record_date,
        ord_class AS work_shift,
        ord_sample_name,
        test_name1,
        ord_up_ver,
        CASE 
            WHEN test_test_value REGEXP '^[0-9.]+$' THEN CAST(test_test_value AS DECIMAL(10,2))
            ELSE NULL
        END AS test_value
    FROM dwd.dwd_lims_sample_analysis_f_1h
    WHERE 
        ord_proc_st != 'PROC_ZF' 
        AND ord_accept_org_code = 'FNR'
        AND ord_productioncategory = '榨蔗生产'
        AND ord_sample_name LIKE '%丙膏%'
        AND test_name1 IN ('锤度', '视纯度')
),
paste_c_data AS (
    SELECT 
        company,
        crushing_season,
        record_date AS paste_record_date,
        work_shift AS paste_work_shift,
        MIN(ord_up_ver) AS min_ord_up_ver,
        MAX(CASE WHEN test_name1 = '锤度' THEN test_value END) AS paste_c_brix,
        MAX(CASE WHEN test_name1 = '视纯度' THEN test_value END) AS paste_c_apparent_purity
    FROM raw_data
    GROUP BY company, crushing_season, record_date, work_shift
),
ranked_paste_data AS (
    SELECT 
        *,
        ROW_NUMBER() OVER (
            PARTITION BY company, crushing_season, paste_record_date, paste_work_shift
            ORDER BY min_ord_up_ver ASC
        ) AS rn
    FROM paste_c_data
)
SELECT 
    company,
    crushing_season,
    paste_record_date,
    paste_work_shift,
    paste_c_brix,
    paste_c_apparent_purity
FROM ranked_paste_data
WHERE rn = 1;
"""
lims_df = spark.sql(lims_sql)

# 读取LIMS样品榨蔗桔水分析数据
lims_sql_js = """
WITH raw_data AS (
    SELECT 
        ord_accept_org_code AS company,
        ord_crushing_season AS crushing_season,
        ord_accept_date AS record_date,
        ord_class AS work_shift,
        ord_sample_name,
        test_name1,
        ord_up_ver,
        CASE 
            WHEN test_test_value REGEXP '^[0-9.]+$' THEN CAST(test_test_value AS DECIMAL(10,2))
            ELSE NULL
        END AS test_value
    FROM dwd.dwd_lims_sample_analysis_f_1h
    WHERE 
        ord_proc_st != 'PROC_ZF' 
        AND ord_accept_org_code = 'FNR'
        AND ord_productioncategory = '榨蔗生产'
        AND ord_sample_name LIKE '%榨蔗桔水%'
        AND test_name1 IN ('重力纯度', '还原糖分')
),
syrup_data AS (
    SELECT 
        company AS syrup_company,
        crushing_season AS syrup_crushing_season,
        record_date AS syrup_record_date,
        work_shift AS syrup_work_shift,
        MIN(ord_up_ver) AS min_ord_up_ver,
        MAX(CASE WHEN test_name1 = '重力纯度' THEN test_value END) AS molasses_gravity_purity,
        MAX(CASE WHEN test_name1 = '还原糖分' THEN test_value END) AS molasses_reducing_sugar
    FROM raw_data
    GROUP BY company, crushing_season, record_date, work_shift
),
ranked_syrup_data AS (
    SELECT 
        *,
        ROW_NUMBER() OVER (
            PARTITION BY syrup_company, syrup_crushing_season, syrup_record_date, syrup_work_shift
            ORDER BY min_ord_up_ver ASC
        ) AS rn
    FROM syrup_data
)
SELECT 
    syrup_company,
    syrup_crushing_season,
    syrup_record_date,
    syrup_work_shift,
    molasses_gravity_purity,
    molasses_reducing_sugar
FROM ranked_syrup_data
WHERE rn = 1;
"""
lims_df_js = spark.sql(lims_sql_js)



# 读取lims班报糖膏数据
lims_ctj_sql = """
SELECT 
    season,
    gongsidm,
    rq,
    CASE 
        WHEN banbiedm = '01' THEN '甲班'
        WHEN banbiedm = '02' THEN '乙班'
        WHEN banbiedm = '03' THEN '丙班'
        ELSE banbiedm
    END AS banbiedm,
    brix_yym,
    brix_bt,
    brix_btg,
    apparent_purity_btg
FROM dwd.dwd_lims_sugar_raw_molasses_batch_report_f_1h WHERE gongsidm = 'FNR';
"""
lims_ctj_df = spark.sql(lims_ctj_sql)
lims_ctj_df = lims_ctj_df.withColumnRenamed("season", "ctj_season")

# 读取LIMS班报榨蔗桔水
lims_js_sql ="""
SELECT 
    season,
    gongsidm,
    rq,
    CASE 
        WHEN banbiedm = '01' THEN '甲班'
        WHEN banbiedm = '02' THEN '乙班'
        WHEN banbiedm = '03' THEN '丙班'
        ELSE banbiedm
    END AS banbiedm,
    chanliang,
    ap
FROM dwd.dwd_lims_sugar_cane_juice_batch_report_f_1h 
WHERE 
    gongsidm = 'FNR' 
    AND name = '榨蔗桔水'
    AND chanliang IS NOT NULL
    AND ap IS NOT NULL;
"""
lims_js_df = spark.sql(lims_js_sql)
lims_js_df = lims_js_df.withColumnRenamed("season", "js_season")

# 读取lims班报实绩数据
lims_bbsj_sql = """
SELECT 
season,
tb_gongsidm,
tb_rq,
CASE 
WHEN tb_banbiedm = '01' THEN '甲班'
WHEN tb_banbiedm = '02' THEN '乙班'
WHEN tb_banbiedm = '03' THEN '丙班'
ELSE '未知班别' 
END AS banbie_name,
sj_zhazheliang
FROM dwd.dwd_lims_batch_report_actual_data_f_1h WHERE tb_gongsidm = 'FNR';
"""
lims_bbsj_df = spark.sql(lims_bbsj_sql)
lims_bbsj_df = lims_bbsj_df.withColumnRenamed("season", "bbsj_season")


# 获取MES数据
# 统一日期字段类型
lims_bbsj_df = lims_bbsj_df.withColumn("tb_rq", to_date(col("tb_rq")))
mes_df = mes_df.withColumn("hand_date", to_date(col("hand_date")))

# 执行左连接
mes_lims_df_v1 = lims_bbsj_df.join(
    mes_df,
    on=[
        lims_bbsj_df["bbsj_season"] == mes_df["season"],
        lims_bbsj_df["tb_gongsidm"] == mes_df["factory"],
        lims_bbsj_df["tb_rq"] == mes_df["hand_date"],
        lims_bbsj_df["banbie_name"] == mes_df["team"]
    ],
    how="left"
)

# 筛选指定保留的字段
mes_lims_df_v1 = mes_lims_df_v1.select(
    col("bbsj_season"),
    col("tb_gongsidm"),
    col("tb_rq"),
    col("banbie_name"),
    col("sj_zhazheliang"),
    col("raw_material_vol_m3"),
    col("sugar_output_vol_m3"),
    col("seed_vol_m3")
)


# 读取lims班报糖膏数据
lims_ctj_df = lims_ctj_df.withColumn("rq", to_date(col("rq")))
mes_lims_df_v1 = mes_lims_df_v1.withColumn("tb_rq", to_date(col("tb_rq")))

mes_lims_df_v2 = mes_lims_df_v1.join(
    lims_ctj_df,
    on=[
        mes_lims_df_v1["bbsj_season"] == lims_ctj_df["ctj_season"],
        mes_lims_df_v1["tb_gongsidm"] == lims_ctj_df["gongsidm"],
        mes_lims_df_v1["tb_rq"] == lims_ctj_df["rq"],
        mes_lims_df_v1["banbie_name"] == lims_ctj_df["banbiedm"]
    ],
    how="left"
)


# 读取LIMS班报榨蔗桔水
lims_js_df = lims_js_df.withColumn("rq", to_date(col("rq")))
mes_lims_df_v2 = mes_lims_df_v2.withColumn("tb_rq", to_date(col("tb_rq")))
mes_lims_df_v3 = mes_lims_df_v2.join(
    lims_js_df,
    on=[
        mes_lims_df_v2["bbsj_season"] == lims_js_df["js_season"],
        mes_lims_df_v2["tb_gongsidm"] == lims_js_df["gongsidm"],
        mes_lims_df_v2["tb_rq"] == lims_js_df["rq"],
        mes_lims_df_v2["banbie_name"] == lims_js_df["banbiedm"]
    ],
    how="left"
)


# 添加计算字段
# 导入必要依赖（若未导入）
from pyspark.sql.functions import col, when

# 基础保留字段
keep_columns = [
    "bbsj_season",
    "tb_gongsidm",
    "tb_rq",
    "banbie_name",
    "sj_zhazheliang",
    "raw_material_vol_m3",
    "brix_yym",
    "seed_vol_m3",
    "brix_bt",
    "sugar_output_vol_m3",
    "brix_btg",
    "apparent_purity_btg",
    "ap",
    "chanliang",
]

# 选择基础字段
mes_lims_df_v3 = mes_lims_df_v3.select(*keep_columns)

# 计算C种与B蜜比例（避免除0）
mes_lims_df_v3 = mes_lims_df_v3.withColumn(
    "seed_b_molasses_ratio",
    when(col("raw_material_vol_m3") != 0, col("seed_vol_m3") / col("raw_material_vol_m3")).otherwise(None)
)

# 计算C糖膏固溶物量T
mes_lims_df_v3 = mes_lims_df_v3.withColumn(
    "c_massecuite_solid_content_t",
    col("sugar_output_vol_m3") * 1.55 * col("brix_btg")*0.001
)

# 计算C糖膏固溶物对蔗比（避免除0）
mes_lims_df_v3 = mes_lims_df_v3.withColumn(
    "c_massecuite_solid_to_cane_ratio",
    when(col("sj_zhazheliang") != 0, col("sugar_output_vol_m3") / col("sj_zhazheliang")).otherwise(None)
)

# 计算C糖膏蜜纯度差
mes_lims_df_v3 = mes_lims_df_v3.withColumn(
    "c_massecuite_molasses_purity_diff",  # C糖膏蜜纯度差
    col("apparent_purity_btg") - col("ap")
)

# 按指定顺序排列字段（C糖膏蜜纯度差放最后）
final_columns = [
    "bbsj_season",
    "tb_gongsidm",
    "tb_rq",
    "banbie_name",
    "sj_zhazheliang",
    "raw_material_vol_m3",
    "brix_yym",
    "seed_vol_m3",
    "brix_bt",
    "seed_b_molasses_ratio",  # C种与B蜜比例
    "sugar_output_vol_m3",
    "c_massecuite_solid_to_cane_ratio",  # C糖膏固溶物对蔗比（brix_btg前）
    "brix_btg",
    "apparent_purity_btg",
    "ap",
    "chanliang",
    "c_massecuite_solid_content_t",  # C糖膏固溶物量T
    "c_massecuite_molasses_purity_diff"  # C糖膏蜜纯度差（放最后）
]

# 按顺序选择字段并展示
mes_lims_df_v3 = mes_lims_df_v3.select(*final_columns)


from pyspark.sql.functions import (
    col, when, to_date, sum, count, round, lit, avg
)
from pyspark.sql.window import Window

# 连接LIMS样品分析丙膏数据
lims_df = lims_df.withColumn("paste_record_date", to_date(col("paste_record_date")))
lims_df_js = lims_df_js.withColumn("syrup_record_date", to_date(col("syrup_record_date")))
mes_lims_df_v3 = mes_lims_df_v3.withColumn("tb_rq", to_date(col("tb_rq")))
lims_df_all = lims_df.join(
    lims_df_js,
    on=[
        lims_df["crushing_season"] == lims_df_js["syrup_crushing_season"],  # 榨季
        lims_df["company"] == lims_df_js["syrup_company"],          # 公司
        lims_df["paste_record_date"] == lims_df_js["syrup_record_date"],      # 日期
        lims_df["paste_work_shift"] == lims_df_js["syrup_work_shift"]  # 班组
    ],
    how="left"
)

mes_lims_df_v4 = mes_lims_df_v3.join(
    lims_df_all,
    on=[
        mes_lims_df_v3["bbsj_season"] == lims_df_all["crushing_season"],  # 榨季
        mes_lims_df_v3["tb_gongsidm"] == lims_df_all["company"],          # 公司
        mes_lims_df_v3["tb_rq"] == lims_df_all["paste_record_date"],      # 日期
        mes_lims_df_v3["banbie_name"] == lims_df_all["paste_work_shift"]  # 班组
    ],
    how="left"
)
final_columns = [
    "bbsj_season",                # 榨季
    "tb_gongsidm",                # 公司
    "tb_rq",                      # 日期
    "banbie_name",                # 班组
    "raw_material_vol_m3",        # B蜜M3
    "brix_yym",                   # B蜜锤度BX
    "seed_vol_m3",                # C种M3
    "brix_bt",                    # C种锤度BX
    "seed_b_molasses_ratio",      # C糖种与B蜜比例
    "sugar_output_vol_m3",        # C糖膏M3
    "c_massecuite_solid_to_cane_ratio",  # C糖膏固溶物对蔗比
    "brix_btg",                   # C糖膏锤度Bx（97～103）
    "apparent_purity_btg",        # C糖膏纯度AP（49～55）
    "ap",                         # C蜜纯度AP
    "c_massecuite_solid_content_t",  # C糖膏固溶物量T
    "c_massecuite_molasses_purity_diff",  # C糖膏蜜纯度差
    "paste_c_brix",               # C糖膏锤度
    "paste_c_apparent_purity",    # C糖膏表观纯度
    "molasses_gravity_purity",    # 桔水重力纯度GP
    "molasses_reducing_sugar",    # 桔水还原糖分
    "chanliang",                  # 桔水产量
    "sj_zhazheliang",             # 榨蔗量T
]

# 筛选最终字段（辅助字段自动剔除）
mes_lims_df_v4 = mes_lims_df_v4.select(*final_columns)


from pyspark.sql import Window
from pyspark.sql.functions import col, when, lit, sum, count, round

# ===================== 1. 计算桔水产率（避免除0异常） =====================
mes_lims_df_v4 = mes_lims_df_v4.withColumn(
    "juice_yield_rate",  # 桔水产率
    when(col("sj_zhazheliang") != 0, col("chanliang") / col("sj_zhazheliang")).otherwise(None)
)

# ===================== 2. 【修正版】C糖膏锤度、纯度合格标识 =====================
# 逻辑：只有非空且在范围内才算合格，非空但不在范围内算不合格，空值保持为空
mes_lims_df_v4 = mes_lims_df_v4.withColumn(
    "c_massecuite_brix_qualified",  # C糖膏锤度合格标识（1=合格，0=不合格，NULL=未检测）
    when(col("paste_c_brix").isNotNull() & 
         (col("paste_c_brix") >= 97) & 
         (col("paste_c_brix") <= 103), lit(1))
    .when(col("paste_c_brix").isNotNull(), lit(0))
).withColumn(
    "c_massecuite_ap_qualified",  # C膏纯度合格标识（1=合格，0=不合格，NULL=未检测）
    when(col("paste_c_apparent_purity").isNotNull() & 
         (col("paste_c_apparent_purity") >= 49) & 
         (col("paste_c_apparent_purity") <= 55), lit(1))
    .when(col("paste_c_apparent_purity").isNotNull(), lit(0))
)

# ===================== 3. 班次排序（必须在累计前定义） =====================
mes_lims_df_v4 = mes_lims_df_v4.withColumn(
    "shift_sort",
    when(col("banbie_name") == "甲班", 1)
    .when(col("banbie_name") == "乙班", 2)
    .when(col("banbie_name") == "丙班", 3)
    .otherwise(4)
)

# ===================== 4. 定义累计窗口 =====================
cum_win = Window.partitionBy("bbsj_season", "tb_gongsidm")\
                .orderBy("tb_rq", "shift_sort")\
                .rowsBetween(Window.unboundedPreceding, Window.currentRow)

# ===================== 5. 【修正版】计算累计合格率 =====================
mes_lims_df_v5 = mes_lims_df_v4.withColumn(
    # 锤度相关累计
    "cum_brix_qty", sum("c_massecuite_brix_qualified").over(cum_win),  # 本榨季累计锤度合格数
).withColumn(
    "cum_brix_total", count("c_massecuite_brix_qualified").over(cum_win)  # 本榨季累计锤度检测数
).withColumn(
    "cum_brix_qualify_rate",  # 本榨季累计锤度合格率
    when(col("cum_brix_total") != 0, 
         round(col("cum_brix_qty") / col("cum_brix_total"), 4)
    ).otherwise(None)
    
    # 纯度相关累计
).withColumn(
    "cum_ap_qty", sum("c_massecuite_ap_qualified").over(cum_win)  # 本榨季累计纯度合格数
).withColumn(
    "cum_ap_total", count("c_massecuite_ap_qualified").over(cum_win)  # 本榨季累计纯度检测数
).withColumn(
    "cum_ap_qualify_rate",  # 本榨季累计纯度合格率
    when(col("cum_ap_total") != 0, 
         round(col("cum_ap_qty") / col("cum_ap_total"), 4)
    ).otherwise(None)
)

# ===================== 6. 最终字段筛选（保留注释） =====================
final_columns = [
    "bbsj_season",                # 榨季
    "tb_gongsidm",                # 公司
    "tb_rq",                      # 日期
    "banbie_name",                # 班组
    "raw_material_vol_m3",        # B蜜M3
    "brix_yym",                   # B蜜锤度BX
    "seed_vol_m3",                # C种M3
    "brix_bt",                    # C种锤度BX
    "seed_b_molasses_ratio",      # C种与B蜜比例
    "sugar_output_vol_m3",        # C糖膏M3
    "c_massecuite_solid_to_cane_ratio",  # C糖膏固溶物对蔗比
    "brix_btg",                   # C糖膏锤度Bx（97～103）
    "apparent_purity_btg",        # C糖膏纯度AP（49～55）
    "ap",                         # C蜜纯度AP
    "c_massecuite_solid_content_t",  # C糖膏固溶物量T
    "c_massecuite_molasses_purity_diff",  # C糖膏蜜纯度差
    "paste_c_brix",               # 样品分析C糖膏锤度
    "paste_c_apparent_purity",    # 样品分析C糖膏表观纯度
    "cum_brix_qualify_rate",      # 累计：C糖膏锤度合格率
    "cum_ap_qualify_rate",        # 累计：C糖膏纯度合格率
    "molasses_gravity_purity",    # 桔水重力纯度GP
    "molasses_reducing_sugar",    # 桔水还原糖分
    "chanliang",                  # 桔水产量
    "juice_yield_rate",           # 桔水产率
    "sj_zhazheliang",             # 榨蔗量T
]

mes_lims_df_v5 = mes_lims_df_v5.select(*final_columns)

from pyspark.sql.types import (
    StructType, StructField, StringType, DateType, 
    DecimalType, DoubleType, TimestampType
)
from pyspark.sql.functions import col, lit
from datetime import datetime

# ===================== 写入Hive表 - C糖膏版本 =====================
# 1. 定义目标表名
target_table = "app.app_mes_c_syrup_continuous_cook_control_params_team_stats_f_1h"

# 2. 定义目标Schema
target_schema = StructType([
    StructField("bbsj_season", StringType(), True),
    StructField("tb_gongsidm", StringType(), True),
    StructField("tb_rq", DateType(), True),
    StructField("banbie_name", StringType(), True),
    StructField("raw_material_vol_m3", DecimalType(10,2), True),
    StructField("brix_yym", DecimalType(32,10), True),
    StructField("seed_vol_m3", DecimalType(10,2), True),
    StructField("brix_bt", DecimalType(32,10), True),
    StructField("seed_b_molasses_ratio", DecimalType(23,13), True),
    StructField("sugar_output_vol_m3", DecimalType(10,2), True),
    StructField("c_massecuite_solid_to_cane_ratio", DecimalType(38,20), True),
    StructField("brix_btg", DecimalType(32,10), True),
    StructField("apparent_purity_btg", DecimalType(32,10), True),
    StructField("ap", DecimalType(32,10), True),
    StructField("c_massecuite_solid_content_t", DoubleType(), True),
    StructField("c_massecuite_molasses_purity_diff", DecimalType(33,10), True),
    
    StructField("paste_c_brix",  DecimalType(32,10), True),
    StructField("paste_c_apparent_purity",  DecimalType(32,10), True),
    
    StructField("cum_brix_qualify_rate",  DecimalType(32,10), True),
    StructField("cum_ap_qualify_rate",  DecimalType(32,10), True),
    StructField("molasses_gravity_purity", DecimalType(32,10), True),
    StructField("molasses_reducing_sugar", DecimalType(32,10), True),
    StructField("chanliang", DecimalType(10,2), True),
    StructField("juice_yield_rate", DecimalType(38,20), True),
    StructField("sj_zhazheliang", DecimalType(30,10), True),
    StructField("source_flg", StringType(), True),
    StructField("dw_update_time", TimestampType(), True)
])

# 3. 补充缺失字段
mes_lims_df_v5 = mes_lims_df_v5.withColumn("source_flg", lit("LIMS+MES")) \
                               .withColumn("dw_update_time", lit(datetime.now()))

# 4. 将DataFrame的每一列转换为目标类型（避免使用rdd转换）
select_exprs = []
for field in target_schema.fields:
    # 如果字段已在DataFrame中，cast到目标类型；否则（理论上不会发生）使用lit(None)
    if field.name in mes_lims_df_v5.columns:
        select_exprs.append(col(field.name).cast(field.dataType).alias(field.name))
    else:
        # 对于缺失字段，补充null并cast（此处source_flg和dw_update_time已添加，其他字段应都存在）
        select_exprs.append(lit(None).cast(field.dataType).alias(field.name))

df_to_write = mes_lims_df_v5.select(*select_exprs)

# 5. 写入Hive表
df_to_write.write \
    .mode("overwrite") \
    .format("hive") \
    .option("encoding", "UTF-8") \
    .saveAsTable(target_table)

# 6. 添加表级注释
spark.sql(f"""
    ALTER TABLE {target_table} 
    SET TBLPROPERTIES (
        'comment' = 'C糖煮制(班报)',
        'create_time' = '{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}'
    )
""")

# 7. 定义字段注释
field_comments = {
    "bbsj_season": "榨季",
    "tb_gongsidm": "公司代码",
    "tb_rq": "日期",
    "banbie_name": "班别",
    "raw_material_vol_m3": "B蜜量M3",
    "brix_yym": "B蜜锤度BX",
    "seed_vol_m3": "C种量M3",
    "brix_bt": "C种锤度BX",
    "seed_b_molasses_ratio": "C种与B蜜比例",
    "sugar_output_vol_m3": "C糖膏M3",
    "c_massecuite_solid_to_cane_ratio": "C糖膏立方数对蔗比",
    "brix_btg": "C糖膏锤度Bx 97～103",
    "apparent_purity_btg": "C糖膏纯度AP 49～55",
    "ap": "C蜜纯度AP",
    "c_massecuite_solid_content_t": "C糖膏固溶物量T",
    "c_massecuite_molasses_purity_diff": "C糖膏蜜纯度差",
    
    "paste_c_brix": "样品分析C糖膏锤度",
    "paste_c_apparent_purity": "样品分析C膏纯度",
    
    "cum_brix_qualify_rate": "C糖膏锤度合格率",
    "cum_ap_qualify_rate": "C膏纯度合格率",
    "molasses_gravity_purity": "桔水重力纯度GP",
    "molasses_reducing_sugar": "桔水还原糖分",
    "chanliang": "桔水产量",
    "juice_yield_rate": "桔水产率",
    "sj_zhazheliang": "榨蔗量T",
    "source_flg": "数据来源标识（LIMS+MES）",
    "dw_update_time": "数仓更新时间"
}

# 8. 工具函数：转换Spark类型为Hive类型字符串
def get_hive_type_str(field):
    data_type = field.dataType
    type_name = type(data_type).__name__
    if type_name == "StringType":
        return "STRING"
    elif type_name == "DateType":
        return "DATE"
    elif type_name == "TimestampType":
        return "TIMESTAMP"
    elif type_name == "DoubleType":
        return "DOUBLE"
    elif type_name == "DecimalType":
        return f"DECIMAL({data_type.precision},{data_type.scale})"
    else:
        return "STRING"

# 9. 批量添加字段注释
field_type_map = {f.name: get_hive_type_str(f) for f in target_schema.fields}
for field, comment in field_comments.items():
    spark.sql(f"""
        ALTER TABLE {target_table} 
        CHANGE COLUMN {field} {field} {field_type_map.get(field, "STRING")} COMMENT '{comment}'
    """)

# 10. 输出执行结果
print(f"✅ 数据已成功写入Hive表：{target_table}")
print(f"📝 表注释：C糖煮制(班报) v1，按榨季独立计算全局累计合格率（跨日期连续累计）")
print(f"🕒 写入时间：{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# 可选：停止Spark会话
spark.stop()

Spark 启动中...
✅ 数据已成功写入Hive表：app.app_mes_c_syrup_continuous_cook_control_params_team_stats_f_1h
📝 表注释：C糖煮制(班报) v1，按榨季独立计算全局累计合格率（跨日期连续累计）
🕒 写入时间：2026-03-11 16:17:06
